In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch

from helpers.data_management import create_LOSO_dataset_dataloader, save_checkpoint, load_model
from helpers.constants import *
from helpers.visualization import inspect_knee_moment_dataset, plot_loss, prediction_overlay
from helpers.modules import KneeCNN
from helpers.running import train_model, evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
leave_one_out_subject = SUBJECTS[2]
batch_size = 32
train_dataset, test_dataset, train_dataloader, test_dataloader = \
    create_LOSO_dataset_dataloader(
        leave_one_out_subject, PERIODIC_TASK_PREFIXES, batch_size=batch_size)

print(len(train_dataset))
print(len(test_dataset))

# dataset = KneeMomentDataset([SUBJECTS[2]],["normal_walk"], stride=5)
# # not LOSO split example
# split = int(0.8 * len(dataset))
# train_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(range(split)))
# val_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(range(split, len(dataset))))

iterator = iter(train_dataset) # only for visualization

In [ ]:
# visualize dataset
x,y = next(iterator)
_,_ = inspect_knee_moment_dataset(torch.squeeze(x),y)


In [ ]:
cnn_model = KneeCNN(in_channels=14).to(device)
checkpoint = train_model(cnn_model, train_dataloader, test_dataloader, num_epochs=10)

plot_loss(checkpoint['train_losses'], checkpoint['test_losses'])
save_checkpoint(checkpoint, 'cnn_model')

In [ ]:
preds, targets, rmse, r2 = evaluate_model(cnn_model, test_dataloader, train_dataset.scaler_y)
print(f'RMSE: {rmse:.4f} Nm')
print(f'R\u00b2:   {r2:.4f}')

prediction_overlay(targets, preds, rmse, r2)

In [ ]:
new_cnn = KneeCNN(in_channels=14).to(device)
load_model(new_cnn, "cnn_model")

In [ ]:
from helpers.modules import init_model_params
init_model_params(new_cnn)

preds, targets, rmse, r2 = evaluate_model(new_cnn, test_dataloader, train_dataset.scaler_y)
print(f'RMSE: {rmse:.4f} Nm')
print(f'R\u00b2:   {r2:.4f}')

prediction_overlay(targets, preds, rmse, r2)

In [ ]:
dataset_dataloader = create_LOSO_dataset_dataloader(
        SUBJECTS[1], ["normal_walk"], batch_size=batch_size)

In [ ]:
from helpers.running import loso_cross_validation
loso_cnn = KneeCNN(in_channels=14).to(device)

rmses, r2s = loso_cross_validation(SUBJECTS, ["normal_walk"], loso_cnn, num_epoches = 10)

In [ ]:
print(rmses)
print(r2s)

In [ ]:
ablated_sensor = "imu_sim"
# ablated_sensor = "imu_shank"

# ablated_cnn = KneeCNN(in_channels=14).to(device)
# rmses, r2s = loso_cross_validation(SUBJECTS, ["normal_walk"], ablated_cnn, num_epoches = 5, 
#                                    checkpoint_name = ablated_sensor)

from helpers.data_management import KneeMomentDataset

ablated_dataset = KneeMomentDataset([SUBJECTS[2]], ["normal_walk"], ablated_sensor = ablated_sensor)

In [ ]:
ablated_dataset.feature_cols
ablated_dataset.X.shape